# Subsidized Health Insurance Premium Design Dataset

## Problem statement

A payer is designing a voluntary health-insurance portfolio for a heterogeneous eligible population. For each population segment, it must choose **one premium/subsidy package**.

The decision is difficult because the objectives conflict:

- lower enrollee premiums improve **take-up and financial access** but require more subsidy;
- higher enrollee premiums reduce the subsidy burden but reduce take-up;
- because healthier / lower-cost people are more price-sensitive, higher prices can leave a **more expensive risk pool** (adverse selection);
- high-risk members need protection from being priced out;
- the overall portfolio must remain financially sustainable.

This is therefore a natural **weighted goal linear programming (GLP)** problem: no single solution is expected to maximize coverage, equity and affordability while also minimizing subsidy use and meeting a financial-surplus target.

> **Important:** the numerical values below are a **constructed synthetic example**, not estimates from a specific insurance program. The qualitative relationships are informed by the literature cited below.

## Literature informing the construction

1. **Yeo, Smith, Willis & Brooks (2002)** — *A mathematical programming approach to optimise insurance premium pricing within a data mining framework*, Journal of the Operational Research Society, 53(11), 1197–1203. DOI: 10.1057/palgrave.jors.2601413  
   Methodological precedent for combining risk-group predictions, price sensitivity and mathematical programming to balance profitability and market share.

2. **Finkelstein, Hendren & Shepard (2019)** — *Subsidizing Health Insurance for Low-Income Adults: Evidence from Massachusetts*, American Economic Review, 109(4), 1530–1567. DOI: 10.1257/aer.20171455  
   Provides empirical motivation for strong premium sensitivity among low-income consumers and for adverse selection as enrollee premiums rise.

3. **Malani et al. (2024; forthcoming in AEJ: Economic Policy)** — *Evaluating and Pricing Health Insurance in Lower-Income Countries: A Field Experiment in India*. NBER Working Paper 32239. DOI: 10.3386/w32239  
   India-specific evidence that premiums and subsidies materially affect insurance take-up and that positive prices can generate adverse selection.

4. **Kifmann (2002)** — *Community rating in health insurance and different benefit packages*, Journal of Health Economics, 21(5), 719–737. DOI: 10.1016/S0167-6296(02)00063-2  
   Motivates the equity problem created by risk pooling, premium regulation and subsidies for higher-risk members.


In [ ]:
import pandas as pd

# -------------------------------------------------
# 1. Eligible population segments
# -------------------------------------------------
#
# Constructed example data. Values are synthetic and are not estimates
# from any one empirical study. They are designed to reproduce health-
# insurance trade-offs documented in the cited literature.

segments = pd.DataFrame({
    "Segment": [
        "Informal_Young_LowRisk",
        "Informal_Family_MediumRisk",
        "SelfEmployed_MiddleRisk",
        "Salaried_Family_MediumRisk",
        "Older_HighRisk",
    ],
    "Employment_Type": [
        "Informal",
        "Informal",
        "Self-employed",
        "Salaried",
        "Retired/Older",
    ],
    "Income_Band": [
        "Low",
        "Low",
        "Middle",
        "Middle-High",
        "Low-Middle",
    ],
    "Risk_Band": [
        "Low",
        "Medium",
        "Medium",
        "Medium",
        "High",
    ],
    # Protected low-income groups for equity / affordability goals
    "Vulnerable_Group": [1, 1, 0, 0, 0],
    # High-risk group for access / anti-selection goals
    "High_Risk_Group": [0, 0, 0, 0, 1],
    # Number of people eligible to enrol in the scheme
    "Eligible_Lives": [1200, 1000, 900, 1300, 600],
    # Stylised annual household / enrollee income proxy
    "Annual_Income_Proxy": [
        180000.0,
        240000.0,
        420000.0,
        720000.0,
        300000.0,
    ],
    # Administrative cost incurred for each enrolled life
    "Admin_Cost_per_Enrollee": [
        800.0,
        900.0,
        1000.0,
        1100.0,
        1400.0,
    ],
})

segments


## Candidate premium and subsidy packages

Each option combines an enrollee contribution, a per-enrollee subsidy, predicted take-up and predicted claims cost among those who enrol.

In [ ]:
# -------------------------------------------------
# 2. Candidate premium / subsidy packages
# -------------------------------------------------
#
# The optimiser must select exactly one option for each segment.
#
# Lower enrollee contributions generally:
#   - increase expected take-up,
#   - require more public / cross-subsidy support, and
#   - retain a broader risk pool.
#
# Higher enrollee contributions generally:
#   - reduce subsidy requirements,
#   - reduce take-up, and
#   - increase expected claim cost among remaining enrollees,
#     representing adverse selection (healthier / lower-cost people
#     are more likely to leave as price increases).

price_options = pd.DataFrame({
    "Segment": [
        # Informal young, low-risk
        "Informal_Young_LowRisk",
        "Informal_Young_LowRisk",
        "Informal_Young_LowRisk",
        "Informal_Young_LowRisk",

        # Informal family, medium-risk
        "Informal_Family_MediumRisk",
        "Informal_Family_MediumRisk",
        "Informal_Family_MediumRisk",
        "Informal_Family_MediumRisk",

        # Self-employed, middle-risk
        "SelfEmployed_MiddleRisk",
        "SelfEmployed_MiddleRisk",
        "SelfEmployed_MiddleRisk",
        "SelfEmployed_MiddleRisk",

        # Salaried family, medium-risk
        "Salaried_Family_MediumRisk",
        "Salaried_Family_MediumRisk",
        "Salaried_Family_MediumRisk",
        "Salaried_Family_MediumRisk",

        # Older, high-risk
        "Older_HighRisk",
        "Older_HighRisk",
        "Older_HighRisk",
        "Older_HighRisk",
    ],
    "Option_ID": [
        "IY_A", "IY_B", "IY_C", "IY_D",
        "IF_A", "IF_B", "IF_C", "IF_D",
        "SE_A", "SE_B", "SE_C", "SE_D",
        "SF_A", "SF_B", "SF_C", "SF_D",
        "OH_A", "OH_B", "OH_C", "OH_D",
    ],
    # Annual amount paid directly by the enrollee
    "Enrollee_Premium": [
        1500.0, 2500.0, 3500.0, 4500.0,
        2500.0, 4000.0, 5500.0, 7000.0,
        9000.0, 10500.0, 12000.0, 13500.0,
        13000.0, 14500.0, 16000.0, 18000.0,
        8000.0, 11000.0, 14000.0, 17000.0,
    ],
    # Public / employer / cross-subsidy payment received by the insurer
    # for each person who enrols under the selected option
    "Subsidy_per_Enrollee": [
        5000.0, 4000.0, 3000.0, 2000.0,
        8500.0, 7000.0, 5500.0, 4000.0,
        2500.0, 1500.0, 500.0, 0.0,
        0.0, 0.0, 0.0, 0.0,
        22000.0, 19000.0, 16000.0, 13000.0,
    ],
    # Predicted proportion of eligible lives who enrol at this contribution
    "Expected_TakeUp": [
        0.92, 0.84, 0.72, 0.58,
        0.90, 0.82, 0.70, 0.55,
        0.88, 0.80, 0.69, 0.56,
        0.94, 0.89, 0.80, 0.68,
        0.93, 0.86, 0.74, 0.60,
    ],
    # Expected claim cost among those who enrol under this option.
    # This is deliberately option-specific: as enrollee price increases,
    # the retained pool becomes costlier on average.
    "Expected_Claim_Cost_per_Enrollee": [
        4300.0, 4700.0, 5200.0, 6000.0,
        7800.0, 8500.0, 9400.0, 10500.0,
        10000.0, 10600.0, 11400.0, 12500.0,
        12200.0, 12600.0, 13400.0, 14600.0,
        24000.0, 25500.0, 27500.0, 30000.0,
    ],
})

price_options


In [ ]:
# -------------------------------------------------
# 3. Derived option-level economics
# -------------------------------------------------

option_data = price_options.merge(
    segments[
        [
            "Segment",
            "Eligible_Lives",
            "Annual_Income_Proxy",
            "Admin_Cost_per_Enrollee",
            "Vulnerable_Group",
            "High_Risk_Group",
        ]
    ],
    on="Segment",
    how="left",
)

option_data["Gross_Premium_Received"] = (
    option_data["Enrollee_Premium"]
    + option_data["Subsidy_per_Enrollee"]
)

option_data["Expected_Enrollees"] = (
    option_data["Eligible_Lives"]
    * option_data["Expected_TakeUp"]
)

option_data["Expected_Subsidy_Spend"] = (
    option_data["Expected_Enrollees"]
    * option_data["Subsidy_per_Enrollee"]
)

option_data["Expected_Revenue"] = (
    option_data["Expected_Enrollees"]
    * option_data["Gross_Premium_Received"]
)

option_data["Expected_Claims"] = (
    option_data["Expected_Enrollees"]
    * option_data["Expected_Claim_Cost_per_Enrollee"]
)

option_data["Expected_Admin_Cost"] = (
    option_data["Expected_Enrollees"]
    * option_data["Admin_Cost_per_Enrollee"]
)

option_data["Expected_Surplus"] = (
    option_data["Expected_Revenue"]
    - option_data["Expected_Claims"]
    - option_data["Expected_Admin_Cost"]
)

option_data["Premium_Income_Ratio"] = (
    option_data["Enrollee_Premium"]
    / option_data["Annual_Income_Proxy"]
)

option_data


## Portfolio parameters

In [ ]:
# -------------------------------------------------
# 4. Portfolio constraints and aspirational goals
# -------------------------------------------------

global_params = {
    # HARD CONSTRAINTS / GUARDRAILS

    # Maximum annual public / cross-subsidy budget
    "Subsidy_Budget": 20_000_000.0,

    # The selected portfolio should not be expected to run at a loss
    "Min_Portfolio_Surplus": 0.0,

    # Protected groups cannot be priced above 3% of the income proxy
    "Max_Vulnerable_Premium_Income_Ratio": 0.03,

    # Minimum access floors even when other goals receive higher weights
    "Min_Vulnerable_Coverage_Floor": 0.60,
    "Min_HighRisk_Coverage_Floor": 0.60,

    # ASPIRATIONAL GOALS FOR WEIGHTED GOAL PROGRAMMING

    # Overall population coverage
    "Target_Overall_Coverage": 0.82,

    # Stronger equity target for low-income / informal groups
    "Target_Vulnerable_Coverage": 0.88,

    # Prevent high-risk members from being priced out
    "Target_HighRisk_Coverage": 0.86,

    # Financial sustainability / reserve-building aspiration
    "Target_Portfolio_Surplus": 5_000_000.0,

    # Aspirational average premium burden across vulnerable segments
    "Target_Avg_Vulnerable_Premium_Income_Ratio": 0.017,
}

global_params


## How this dataset maps to a GLP model

### Binary decisions

For every segment \(s\), select exactly one premium/subsidy option \(k\):

\[
\sum_{k \in K_s} y_{sk} = 1
\]

### Hard feasibility constraints

A runnable model can enforce:

- annual subsidy expenditure \(\leq\) the subsidy budget;
- expected portfolio surplus \(\geq 0\);
- a maximum premium-to-income ratio for vulnerable groups;
- minimum coverage floors for vulnerable and high-risk groups.

### Aspirational goals

Weighted GLP can then minimize deviations from:

1. **overall coverage target**;
2. **vulnerable-group coverage target**;
3. **high-risk coverage target**;
4. **portfolio surplus target**;
5. **vulnerable-group affordability target**.

The constructed targets are intentionally conflicting. Under the stated ₹2.0 crore subsidy budget, there is **no premium package combination that simultaneously reaches every aspirational target**. Consequently, changing goal weights changes which policy objective is protected and which target is allowed to deviate — the behaviour a goal-programming example should demonstrate.
